### 0 loading libs & competition data

In [4]:
import pandas as pd 
import numpy as np
%run cfcs.py

In [5]:
riskfutures = pd.read_csv('./corn_climate_risk_futures_daily_master.csv')
marketshare = pd.read_csv('./corn_regional_market_share.csv')
## dir need to change before submission to Kaggle input directories

#### Note: EDAs were done seperately.

### 1 Baseline Feature Engineering

In [6]:
mergedf = riskfutures.copy()
mergedf['day_of_year'] = pd.to_datetime(mergedf['date_on'],format='%Y-%m-%d').dt.dayofyear
mergedf['quarter'] = pd.to_datetime(mergedf['date_on'],format='%Y-%m-%d').dt.quarter

In [7]:
mergedf = mergedf.merge(marketshare[['region_id','percent_country_production']],how='left',on='region_id')

In [8]:
mergedf['percent_country_production'] = mergedf['percent_country_production'].fillna(0.0)
## see EDA_regional_marketshare.ipynb

#### 1.1 Production-Weighted Risk Scores

In [9]:
risk_categories = ['heat_stress', 'unseasonably_cold', 'excess_precip', 'drought']
for risk in risk_categories:
    low = f'climate_risk_cnt_locations_{risk}_risk_low'
    medium = f'climate_risk_cnt_locations_{risk}_risk_medium'
    high = f'climate_risk_cnt_locations_{risk}_risk_high'
    
    risk_scores = (0*mergedf[low]+1*mergedf[medium]+2*mergedf[high])/\
                           (mergedf[low]+mergedf[medium]+mergedf[high])
    ## define regional daily risk score as normalized weighted sum of number of locations
    
    production_weighted_risk_scores = (risk_scores*mergedf['percent_country_production'])/100
    ## use marketshare data to get production-weighted regional daily risk scores
    
    mergedf[f'climate_risk_{risk}_score'] = risk_scores
    mergedf[f'climate_risk_{risk}_weighted_score'] = production_weighted_risk_scores
    ## iterate for all four climate risk types; total 8 new engieered features

#### 1.2 Composite Risk Indices

In [10]:
mergedf['climate_risk_temperature_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories[:2]]].max(axis=1)
## maximum of temperature-related risk scores
mergedf['climate_risk_precipitation_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories[2:]]].max(axis=1)
## maximum of precipitation-related risk scores
mergedf['climate_risk_overall_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories]].max(axis=1)
## maximum of all risk scores
mergedf['climate_risk_avg_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories]].mean(axis=1)
## average of all risk scores
## total 4 new engineered features

#### 1.3 Risk Temporal Summaries

In [11]:
mergedf = mergedf.sort_values(['region_name','date_on'])
window_period = [7,14,30,60,90,120]
## three periods to compute risk scores moving avg and maximum 
for window in window_period:
    for risk in risk_categories:
        mergedf[f'climate_risk_{risk}_ma_{window}d'] = \
        mergedf.groupby(['region_name'])[f'climate_risk_{risk}_score']\
               .rolling(window=window,min_periods=1).mean().reset_index(level=0,drop=True)
## compute risk score moving avg with different windows for different risk types in each region

        mergedf[f'climate_risk_{risk}_max_{window}d'] = \
        mergedf.groupby(['region_name'])[f'climate_risk_{risk}_score']\
               .rolling(window=window,min_periods=1).max().reset_index(level=0,drop=True)
## compute maximum risk scores with different windows for different risk types in each region
## total 6*4*2 = 48 new features

#### 1.4 Risk Momentum

In [12]:
features_change1d = mergedf.groupby('region_name')[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
       .diff(periods=1)\
       .rename(columns=dict(zip([f'climate_risk_{risk}_score' for risk in risk_categories],\
                                [f'climate_risk_{risk}_change_1d' for risk in risk_categories])))
## Daily Change of risk scores for each risk type in each region 

features_acceleration = features_change1d.diff(periods=1)\
                        .rename(columns=\
                                dict(zip([f'climate_risk_{risk}_change_1d' for risk in risk_categories],\
                                         [f'climate_risk_{risk}_acceleration_1d' for risk in risk_categories])))
## Acceleration of daily Change of risk scores for each risk type in each region

features_change1w = mergedf.groupby('region_name')[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
       .diff(periods=7)\
       .rename(columns=dict(zip([f'climate_risk_{risk}_score' for risk in risk_categories],\
                                [f'climate_risk_{risk}_change_1w' for risk in risk_categories])))
## Weekly Change of risk scores for each risk type in each region 

mergedf = pd.concat([mergedf,\
           features_change1d,\
           features_change1w,\
           features_acceleration],axis=1)
## 12 new features in Risk Momentum category

#### 1.5 Cross-Regional features

In [13]:
feature_country = pd.concat([\
mergedf.groupby(['country_name', 'date_on'])\
[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
.agg(['mean','max','std']),
## compute country-wide daily avg, max, and std risk scores
mergedf.groupby(['country_name', 'date_on'])\
[[f'climate_risk_{risk}_weighted_score' for risk in risk_categories]]\
.agg('sum')],axis=1)
## compute country-wide daily production-weighted sum risk scores
feature_country.columns = [f'climate_risk_{risk}_score_country_{metric}'\
                          for risk in risk_categories \
                          for metric in ['mean','max','std']]+\
                          [f'climate_risk_{risk}_weighted_score_country_sum'\
                          for risk in risk_categories]
## rename new features
mergedf = mergedf.merge(feature_country.reset_index(),\
              how='left',\
              on=['country_name','date_on'])
## add 4*4=16 new features

#### 1.6 Baseline features CFCS score

In [9]:
corrdf  = compute_partial_correlations(mergedf)
report = sigcorr_report(corrdf)

In [10]:
report.sort_values('avg_sig_corr',ascending=False).head(10)
## report shows that only few features contain meaningful amount
## of significant correlations

,avg_sig_corr,max_sig_corr,sig_corr_count,sig_corr_ratio(%)
climate_variable,,,,
climate_risk_drought_ma_90d,0.605,0.777,58,2.585
climate_risk_drought_ma_60d,0.605,0.734,51,2.273
climate_risk_drought_ma_120d,0.600,0.791,65,2.897
climate_risk_drought_ma_30d,0.596,0.724,40,1.783
climate_risk_excess_precip_max_7d,0.590,0.590,1,0.045
climate_risk_drought_max_7d,0.578,0.578,1,0.045
climate_risk_heat_stress_ma_90d,0.574,0.641,6,0.333
climate_risk_excess_precip_max_14d,0.573,0.595,2,0.089
climate_risk_excess_precip_ma_14d,0.557,0.597,2,0.089


In [11]:
features_sig_1 = report[report.avg_sig_corr>=0.6].index
## selection features with absolute correlation bigger than 0.6
sigdf = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
lambda x:( not x.startswith('climate_risk')) or (x in features_sig_1)
)]]
cfcs(compute_partial_correlations(sigdf))
## compute cfcs score using backtesting functionalities

2.58% of all correlations are significant
Average significant correlation is 0.603
highest absolute correlation found is 0.791
final CFCS score is 54.42


{'cfcs_score': 54.41746220726535,
 'avg_sig_score': 60.321856321839086,
 'max_corr_score': 79.132,
 'sig_count_score': 2.5846702317290555}

### 2. Advanced Techniques

#### 2.1 Non-linear transformations

##### Trignometic transformations

In [12]:
features_trig = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
        lambda x: x.startswith('climate_risk'))]].transform(['sin','cos','tan'])
## compute sin,cos, and tan for each climate risk feature
## to account for the seasonalities in futures data
features_trig.columns = [f[0]+'_'+f[1] for f in features_trig.columns]
mergedf = pd.concat([mergedf,features_trig],axis=1)


##### Threshold transformations

In [13]:
std_multiplier = [1,2]
features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                     lambda x: x.startswith('climate_risk'))]]

for multiplier in std_multiplier:
    features_threshold = features.transform(lambda x: np.where(x>multiplier*x.std(),x,0))\
                                 .rename(columns=dict(zip(features.columns,\
                                        [f+f'_above_{multiplier}_std'for f in features.columns])))
    ## threshold define in terms of std of each risk feature
    ## retain only values in each feature that exceed the threshold
    mergedf = pd.concat([mergedf,features_threshold],axis=1)

#### 2.2 lag analysis

In [14]:
mergedf = mergedf.sort_values(['region_name','date_on'])
window_period = [7,14,30]
features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                     lambda x: x.startswith('climate_risk'))]]
for window in window_period:
    features_lag = features.shift(periods=window)
    ## generate historial risk features with different periods
    features_lag = features_lag.rename(columns=dict(zip(features_lag.columns,
                       [c+f'_lag_{window}d'for c in features_lag.columns])\
                      ))
    mergedf = pd.concat([mergedf,features_lag],axis=1)

MemoryError: Unable to allocate 3.86 GiB for an array with shape (1614, 320661) and data type float64

#### 2.3 CFCS analysis part.2

In [14]:
corrdf  = compute_partial_correlations(mergedf)
## compute correlation tables; takes long time to finish

In [86]:
report = sigcorr_report(corrdf,sig_level=0.5)
## generate feature significance report

In [87]:
report.sort_values(['max_sig_corr','avg_sig_corr','sig_corr_count'],ascending=False).head(5)
## 'climate_risk_drought_ma_60d_cos_lag_30d' feature contains the highest partial correlation
## found. It's also the one without fewest transfromations compared to other sig. features.
feature_max_sig = 'climate_risk_drought_ma_60d_cos_lag_30d'

In [94]:
report.sort_values(['avg_sig_corr','sig_corr_count'],ascending=False).head(10)
## we want to find a collection of features with high avg_sig_corr and meaningfuk
## amount of correlations.
features_avg_sig = report[(report.avg_sig_corr>0.61)&(report['sig_corr_ratio(%)']>2)].index

In [95]:
features_sig_2 = np.unique(list(features_avg_sig)+[feature_max_sig])
sigdf = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
lambda x:( not x.startswith('climate_risk')) or (x in features_sig_2)
)]]
cfcs(compute_partial_correlations(sigdf))

2.57% of all correlations are significant
Average significant correlation is 0.615
highest absolute correlation found is 0.811
final CFCS score is 55.61


{'cfcs_score': 55.60623703951278,
 'avg_sig_score': 61.52573611111111,
 'max_corr_score': 81.10000000000001,
 'sig_count_score': 2.5668449197860963}

In [115]:
submissiondf = sigdf[sigdf.ID.isin(validation_arr)]
## Helios only accepts certain amount of records for submission
submissiondf.to_csv('./submission_kaggle.csv',index=False)